# Credit Union Quarterly Data Ingestion. This notebook executes the full-coverage ingestion pipeline in `ingest_ncua_call_report.py`.

In [0]:
from ingest_ncua_call_report import run_ingestion

In [0]:
# Configure your ingestion window (inclusive).# Example: start_year=1994 and end_year=2026 requests all quarters in that range.start_year = 1994end_year = 2026output_file = "NCUA_Call_Report.csv"multirecord_output_file = "NCUA_Call_Report_Multirecord.csv"run_ingestion(    start_year=start_year,    end_year=end_year,    output_file=output_file,    multirecord_output_file=multirecord_output_file,    cleanup_temp_files=True,)

In [0]:
from functools import reduce
import time

dfs = []
for year in range(1994, 2027):
    output_file = f"NCUA_Call_Report_{year}.csv"
    multirecord_output_file = f"NCUA_Call_Report_Multirecord_{year}.csv"
    run_ingestion(
        start_year=year,
        end_year=year,
        output_file=f"/Volumes/workspace/ncua/ncua_files/{output_file}",
        multirecord_output_file=f"/Volumes/workspace/ncua/ncua_files/{multirecord_output_file}",
        cleanup_temp_files=True,
    )
    time.sleep(0.25)
    df_year = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"/Volumes/workspace/ncua/ncua_files/{output_file}")
    )
    dfs.append(df_year)

df_union = reduce(lambda a, b: a.unionByName(b), dfs)
display(df_union)

In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Workspace/Users/lpakola@westmonroe.com/CU-Data-Ingestion/NCUA_Call_Report.csv")
)

df.display()

In [0]:
import re

def clean_col(name):
    if " - " in name:
        name = name.split(" - ")[1]

    name = name.lower()
    name = re.sub(r"[ ,;{}()\n\t=/]+", "_", name)
    name = name.replace("-", "_")
    name = re.sub(r"_+", "_", name).strip("_")

    return name

def make_unique(cols):
    seen = {}
    new_cols = []

    for c in cols:
        if c not in seen:
            seen[c] = 0
            new_cols.append(c)
        else:
            seen[c] += 1
            new_cols.append(f"{c}_{seen[c]}")

    return new_cols

cleaned = [clean_col(c) for c in df.columns]
unique_cols = make_unique(cleaned)

df_clean = df.toDF(*unique_cols)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ncua")

In [0]:
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.ncua.ncua_bronze"))

In [0]:
spark.table("workspace.ncua.ncua_bronze").display()

In [0]:
display(spark.sql("SELECT DISTINCT cycle_date FROM workspace.ncua.ncua_bronze ORDER BY cycle_date"))